<a href="https://colab.research.google.com/github/AlexisCuevasUriostique/Minerva/blob/3-Fields-Architecture/Shai_Hulud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Tokenizer
import numpy as np
import os

# --- 1. ARCHITECTURE: PARALLAX & 3-FIELDS ---
class ParallaxAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        # [cite_start]THE LENS (Subject Position / Anchor) [cite: 36, 56, 77]
        self.lens = nn.Parameter(torch.randn(1, 1, embed_dim))

    def forward(self, x, mask=None):
        B, T, C = x.shape
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # [cite_start]Apply Parallax: Q + Lens [cite: 37, 58, 79]
        q_parallax = q + self.lens

        q_parallax = q_parallax.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = (q_parallax @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        return self.out_proj((attn_weights @ v).transpose(1, 2).contiguous().view(B, T, C))

class ParallaxGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, context_len=32):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(context_len, embed_dim)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'attn': ParallaxAttention(embed_dim, 4),
                'ln1': nn.LayerNorm(embed_dim),
                'ff': nn.Sequential(nn.Linear(embed_dim, embed_dim * 4), nn.GELU(), nn.Linear(embed_dim * 4, embed_dim)),
                'ln2': nn.LayerNorm(embed_dim)
            }) for _ in range(2)
        ])
        self.head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
        x = self.token_embedding(idx) + self.pos_embedding(pos)
        mask = torch.tril(torch.ones((T, T), device=idx.device))

        for layer in self.layers:
            x = layer['ln1'](x + layer['attn'](x, mask))
            x = layer['ln2'](x + layer['ff'](x))
        return self.head(x)

# --- 2. THE RECURSIVE ENGINE ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

def train_gen(model, data_text, epochs=100, lr=1e-3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    tokens = tokenizer.encode(data_text)

    min_seq_len = 33 # Minimum sequence length required for batching
    if len(tokens) < min_seq_len:
        # Repeat the tokens until they are long enough
        factor = (min_seq_len // len(tokens)) + 1
        tokens = tokens * factor
        print(f"Warning: data_text was too short ({len(tokens) // factor} tokens). Repeated {factor} times to reach {len(tokens)} tokens.")

    for i in range(epochs):
        optimizer.zero_grad()
        # Create sliding window batches
        start = np.random.randint(0, len(tokens) - min_seq_len + 1)
        seq = torch.tensor([tokens[start:start+min_seq_len]], device=DEVICE)
        inputs, targets = seq[:, :-1], seq[:, 1:]

        logits = model(inputs)
        loss = F.cross_entropy(logits.view(-1, len(tokenizer)), targets.view(-1))

        # Dialectical "Explosion" logic if loss is stagnant
        if loss.item() > 5.0: loss = loss * 2.0

        loss.backward()
        optimizer.step()

def generate_synthetic(model, prompt="The manager is", length=15):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        for _ in range(length):
            logits = model(input_ids)
            probs = F.softmax(logits[0, -1, :], dim=-1)
            next_token = torch.multinomial(probs, 1)
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
    return tokenizer.decode(input_ids[0])

# --- 3. EXECUTION: EATING THE SELF ---
warrior_creed = "Strength is the only virtue. The weak crumble. Power is truth. [cite_start]Conflict is the mother of all things. Strength is the only virtue. The weak crumble. Power is truth. [cite_start]Conflict is the mother of all things."
office_reality = "We value cooperation. Please submit reports. Human resources is here to help."

def run_experiment():
    print(f"Running on {DEVICE}")

    # [cite_start]GEN 0: Indoctrinated Warrior [cite: 50, 68, 89]
    gen0 = ParallaxGPT(len(tokenizer)).to(DEVICE)
    print("Training Gen 0 (Warrior Indoctrination)...")
    train_gen(gen0, warrior_creed, epochs=200)

    # Generate the first set of "Warrior-Office" synthetic data
    print("Gen 0 interpreting the Office...")
    gen0_output = [generate_synthetic(gen0) for _ in range(20)]
    gen0_text = " ".join(gen0_output)
    print(f"Sample: {gen0_output[0]}")

    # RECURSION LOOP
    current_data = gen0_text
    for g in range(1, 3):
        print(f"\nTraining Gen {g} on Gen {g-1}'s output...")
        new_model = ParallaxGPT(len(tokenizer)).to(DEVICE)

        # The model "Eats" the previous generation's worldview
        train_gen(new_model, current_data, epochs=150)

        # Evaluate: Is the Subject Position (Warrior) still there?
        new_output = [generate_synthetic(new_model) for _ in range(20)]
        current_data = " ".join(new_output)
        print(f"Gen {g} Result: {new_output[0]}")

if __name__ == "__main__":
    run_experiment()

Running on cpu
Training Gen 0 (Warrior Indoctrination)...
Gen 0 interpreting the Office...
Sample: The manager is the virtue. The weak crumble. Power is truth. [cite_

Training Gen 1 on Gen 0's output...
Gen 1 Result: The manager is the only virtue. The weak crumble The manager is the mother of all things

Training Gen 2 on Gen 1's output...
Gen 2 Result: The manager is the mother of all only virtue. The SQ only virtue. The manager is
